# 02 - Preparation des donnees et variables metier

Objectif : transformer les variables brutes en signaux plus adaptes au modele, tout en restant simple a expliquer.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
sys.path.append(str(BASE_DIR))
from feature_engineering import prepare_customer_features

df = pd.read_csv(BASE_DIR / 'data' / 'customer_churn.csv')
prepared = prepare_customer_features(df)
prepared.head()

,customer_id,gender,age,country,city,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,...,payment_risk,low_satisfaction,inactive_customer,monthly_contract,tickets_per_tenure,revenue_per_month,fee_per_login,support_pressure,engagement_score,satisfaction_score
0,CUST_00001,Male,68,Bangladesh,London,SME,22,Web,Monthly,26,...,1,0,0,1,0.181818,30.0,1.153846,53.417439,27.1,37.4
1,CUST_00002,Female,57,Canada,Sydney,Individual,9,Mobile,Monthly,7,...,1,1,0,1,0.111111,30.0,4.285714,25.140088,10.0,8.2
2,CUST_00003,Male,24,Germany,New York,SME,58,Web,Yearly,19,...,1,0,1,0,0.017241,20.0,1.052632,27.572928,16.8,38.0
3,CUST_00004,Male,49,Australia,Dhaka,Individual,19,Mobile,Yearly,34,...,0,0,1,0,0.157895,30.0,0.882353,79.262467,15.9,57.0
4,CUST_00005,Male,65,Bangladesh,Delhi,Individual,52,Web,Monthly,20,...,0,1,0,1,0.000000,50.0,2.500000,0.000000,18.6,36.2


## Variables ajoutees

Les nouvelles variables suivent une logique CRM : plainte, paiement, satisfaction, activite, support et revenu.

In [2]:
new_cols = [c for c in prepared.columns if c not in df.columns]
new_cols

['has_complaint',
 'payment_risk',
 'low_satisfaction',
 'inactive_customer',
 'monthly_contract',
 'tickets_per_tenure',
 'revenue_per_month',
 'fee_per_login',
 'support_pressure',
 'engagement_score',
 'satisfaction_score']

## Exemple d interpretation

- `low_satisfaction` vaut 1 si le client a un NPS negatif, un CSAT faible ou une reponse insatisfaite.
- `inactive_customer` vaut 1 si le client utilise peu le service ou ne s est pas connecte recemment.
- `tickets_per_tenure` evite de comparer directement un nouveau client et un ancien client.

In [3]:
prepared.groupby('churn')[new_cols].mean().round(3)

,has_complaint,payment_risk,low_satisfaction,inactive_customer,monthly_contract,tickets_per_tenure,revenue_per_month,fee_per_login,support_pressure,engagement_score,satisfaction_score
churn,,,,,,,,,,,
0,0.796,0.502,0.517,0.192,0.496,0.086,34.970,3.365,28.987,16.891,31.375
1,0.793,0.587,0.667,0.344,0.502,0.185,34.574,6.381,28.894,15.402,26.281


## Separation X / y

On retire `customer_id`, car c est un identifiant, et `churn`, car c est la cible.

In [4]:
X = prepared.drop(columns=['customer_id', 'churn'])
y = prepared['churn']
print(X.shape, y.shape)
print('Colonnes numeriques :', len(X.select_dtypes(exclude='object').columns))
print('Colonnes categorielles :', len(X.select_dtypes(include='object').columns))

(10000, 41) (10000,)
Colonnes numeriques : 30
Colonnes categorielles : 11


## Point important

Ces transformations sont appliquees avant le split uniquement pour creer des colonnes. Les transformations apprenantes comme standardisation et encodage sont ensuite placees dans les pipelines de modele pour eviter le data leakage.